# Photo cards + ball candidates for the whole match

Two things from one pass over the match video:
1. **Photo cards**: 4 crops of every track piece (an unbroken box, nearly always one person), packed into
   one picture per segment. The ID Review page shows them so you can confirm or fix who is who
   without watching the video.
2. **Ball candidates**: the ball model run over the whole picture every 2nd frame. The tracker in the
   full-match run only looked near its last guess, so it drifted; with candidates everywhere, Claude
   re-solves the ball path for the whole match at once.

Output, per segment, in `MyDrive/Playbook/review/match_video_2/<segment>/`: `cards.jpg`, `cards.json`,
`ball_cands.csv.gz`.

**Runtime:** L4 GPU. Cards take about 1 minute per segment, ball candidates about 5 minutes per segment
(about 2 hours in total). **Runtime -> Run all**; do not edit cells. Keep the tab open. If it disconnects,
**Run all again**: finished segments are skipped.

In [ ]:
# Cell 1 — GPU, Drive, inputs
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
GPU = g.stdout.strip(); print('GPU:', GPU)
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
if not os.path.isdir(PBD):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
MATCH = 'match_video_2'
CLIPS = sorted(glob.glob(f'{PBD}/video_downloads/{MATCH}_clips/*.mp4'))
REVIEW = f'{PBD}/review/{MATCH}'
BALL_MODEL = f'{PBD}/ball_model/train/checkpoint_best_total.pth'
NAMES = [os.path.basename(c)[:-4] for c in CLIPS]
missing = [p for p in [BALL_MODEL] + [f'{REVIEW}/{n}/tracks_small.csv.gz' for n in NAMES] if not os.path.exists(p)]
if missing or not CLIPS:
    raise SystemExit(f'Missing in Drive: {missing[:3] or "the match clips"}')
print(len(CLIPS), 'segments:', NAMES[0], '...', NAMES[-1])

In [ ]:
# Cell 2 — Our repo and packages
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
!pip install -q "rfdetr==1.11.0" "supervision>=0.25" "pydantic-settings==2.4.0" "python-dotenv==1.0.1"
import numpy as np, pandas as pd
from tools.id_cards import card_plan, build_sheet, ball_candidates
print('ready')

In [ ]:
# Cell 3 — Photo cards (skips finished segments; ~1 min each)
for clip, n in zip(CLIPS, NAMES):
    if os.path.exists(f'{REVIEW}/{n}/cards.json'):
        continue
    t0 = time.time()
    local = f'/content/{n}.mp4'
    if not os.path.exists(local):
        shutil.copy(clip, local)
    t = pd.read_csv(f'{REVIEW}/{n}/tracks_small.csv.gz')
    plan = card_plan(t, min_frames=30, per_piece=4)
    idx = build_sheet(local, plan, f'/content/cards_{n}.jpg')
    shutil.copy(f'/content/cards_{n}.jpg', f'{REVIEW}/{n}/cards.jpg')
    json.dump(idx, open(f'{REVIEW}/{n}/cards.json', 'w'))
    print(n, len(idx['pieces']), 'pieces', f"{os.path.getsize(f'{REVIEW}/{n}/cards.jpg') / 1e6:.1f} MB", f'{time.time() - t0:.0f}s')
print('cards done')

In [ ]:
# Cell 4 — Ball candidates over the whole picture, every 2nd frame (skips finished segments; ~5 min each)
from vision.ball_model import load_ball_model
fn = None
for clip, n in zip(CLIPS, NAMES):
    dst = f'{REVIEW}/{n}/ball_cands.csv.gz'
    if os.path.exists(dst):
        continue
    if fn is None:
        fn = load_ball_model(BALL_MODEL, 'fp16')
    t0 = time.time()
    local = f'/content/{n}.mp4'
    if not os.path.exists(local):
        shutil.copy(clip, local)
    c = ball_candidates(local, fn, every=2, conf=0.05, top=8)
    c.to_csv('/content/bc.csv.gz', index=False, compression='gzip'); shutil.copy('/content/bc.csv.gz', dst)
    os.remove(local)
    left = sum(not os.path.exists(f'{REVIEW}/{m}/ball_cands.csv.gz') for m in NAMES)
    print(f'{n}: {len(c)} candidates on {c.frame.nunique()} frames, {(time.time() - t0) / 60:.1f} min | {left} segments left')
print(f'\nDone. Tell Claude the cards and ball candidates are in {REVIEW}.')